# A GraphSAGE Approach for Fraud Detection on the IEEE-CIS Dataset
## A Comparative Study with XGBoost on Accuracy and Inference Latency

**Research Question:** *Does an inductive GraphSAGE model exploiting shared-entity edges between transactions achieve materially higher AUC than an XGBoost tabular baseline on the IEEE-CIS dataset, and at what inference-latency cost per 1,000 transactions?*

---
### Notebook Workflow
1. **Configuration & Reproducibility:** Global seeds and hardware setup.
2. **Data Ingestion & Preprocessing:** Strict chronological 80/10/10 split, memory downcasting, leak-free scaling.
3. **Causal Graph Construction:** Frequency-capped anti-supernode heuristics, temporal causality ($t_{src} \le t_{dst}$).
4. **Model Training:** XGBoost, Random Forest, Logistic Regression, GraphSAGE, GAT, and Hybrid.
5. **Benchmarking & Latency Profiling:** Dual latency (Batch 1k & Stream 1 tx), Latency decomposition.
6. **Visualizations:** PR/ROC curves, Pareto trade-off curve, Degree distribution, Cold-start analysis.
7. **Model Persistence:** Native PyTorch/JSON checkpoints + ONNX Runtime export.

## 1. Setup & Deterministic Seeding

In [ ]:
import os
import sys
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Ensure project root is in sys.path
sys.path.append(os.path.abspath("."))

from src.config import CONFIG, seed_everything, get_device
from src.preprocessing import load_and_preprocess
from src.graph_builder import build_causal_transaction_graph, get_graph_statistics
from src.models import train_tabular_baselines, train_graphsage, train_gat
from src.benchmark import (
    compute_classification_metrics,
    predict_gnn_probabilities,
    benchmark_tabular_latency,
    benchmark_gnn_latency,
    evaluate_cold_start_robustness,
    export_results_table
)
from src.visualization import (
    plot_degree_distribution,
    plot_temporal_fraud_distribution,
    plot_pr_and_roc_curves,
    plot_accuracy_latency_tradeoff,
    plot_latency_breakdown,
    plot_cold_start_comparison
)
from src.serialization import save_native_models, export_tabular_to_onnx, train_hybrid_model

# Choose execution mode: "prototype" (100k transactions) or "full" (590k transactions)
config = CONFIG.copy()
config["subset_size"] = 100_000  # Set to None for full 590k dataset

seed_everything(config["seed"])
device = get_device()
print(f"Mode: {'PROTOTYPE (100k)' if config['subset_size'] else 'FULL (590k)'}")

## 2. Leak-Free Preprocessing & Chronological Splitting
- Sorts transactions strictly ascending by `TransactionDT`.
- Partitions data into **80% Train / 10% Validation / 10% Test** without shuffling.
- Strips `TransactionID` to prevent trivial ID memorization.
- Downcasts numeric data types to fit comfortably within 16GB RAM.

In [ ]:
data_dict = load_and_preprocess(config)

# Visualize temporal volume and fraud prevalence over time
plot_temporal_fraud_distribution(data_dict, config)
plt.show()

## 3. Causal Graph Construction (Anti-Supernode Heuristics)
- Edges connect transactions sharing `card1`, `DeviceInfo`, and `P_emaildomain`.
- `DeviceType` is excluded to prevent an $O(N^2)$ combinatorial explosion of 5.1 billion edges.
- Entities with frequency $> 500$ (e.g. generic `gmail.com`) are capped.
- Directed causal edges strictly enforce $t_{src} \le t_{dst}$.

In [ ]:
graph_data = build_causal_transaction_graph(data_dict, config)
graph_stats = get_graph_statistics(graph_data)

# Plot in-degree distribution on log-log scale
plot_degree_distribution(graph_data, config)
plt.show()

## 4. Train Tabular Baselines (XGBoost, Random Forest, Logistic Regression)
- **XGBoost:** Primary GBDT baseline with `scale_pos_weight` and `tree_method='hist'`.
- **Random Forest:** 200 trees with `class_weight='balanced'`.
- **Logistic Regression:** Linear floor with `StandardScaler`.

In [ ]:
tabular_models = train_tabular_baselines(data_dict, config)

## 5. Train Proposed Inductive GraphSAGE
- 2-layer SAGEConv with BatchNorm, Dropout (0.3), and class-weighted BCE loss.
- Mini-batch training via `NeighborLoader` with dynamic 2-hop fan-out $[15, 10]$.
- Gradient clipping and Early Stopping based on validation ROC-AUC.

In [ ]:
graphsage_model = train_graphsage(graph_data, config, device)

## 6. Train Appendix Baseline: Graph Attention Network (GAT)
- 2-layer GAT with 4-head multi-head attention to quantify the attention computational penalty.

In [ ]:
gat_model = train_gat(graph_data, config, device)

## 7. Proposed Advanced Extension: Hybrid Model (GraphSAGE + XGBoost)
- Extracts 128-dimensional topological node embeddings from trained GraphSAGE Layer 2.
- Concatenates embeddings with tabular features: $X_{\text{hybrid}} = [X_{\text{tabular}} \;|\; Z_{\text{embeddings}}]$.
- Trains a secondary XGBoost model on the enriched manifold.

In [ ]:
hybrid_model, X_hybrid = train_hybrid_model(graphsage_model, graph_data, data_dict, config, device)

## 8. Out-of-Sample Performance Evaluation (Table 1: Classification Metrics)
Evaluates ROC-AUC, PR-AUC, F1-Score, and Balanced Accuracy on the held-out test split (10%).

In [ ]:
test_idx = data_dict["test_idx"]
y_test = data_dict["y"][test_idx]
X_tab_test = data_dict["X_tabular"][test_idx]
X_neu_test = data_dict["X_neural"][test_idx]

predictions_dict = {}
metrics_rows = []

# XGBoost
xgb_prob = tabular_models["XGBoost"].predict_proba(X_tab_test)[:, 1]
predictions_dict["XGBoost"] = xgb_prob
metrics_rows.append({"Model": "XGBoost", **compute_classification_metrics(y_test, xgb_prob)})

# Random Forest
rf_prob = tabular_models["RandomForest"].predict_proba(X_tab_test)[:, 1]
predictions_dict["RandomForest"] = rf_prob
metrics_rows.append({"Model": "RandomForest", **compute_classification_metrics(y_test, rf_prob)})

# Logistic Regression
lr_prob = tabular_models["LogisticRegression"].predict_proba(X_neu_test)[:, 1]
predictions_dict["LogisticRegression"] = lr_prob
metrics_rows.append({"Model": "LogisticRegression", **compute_classification_metrics(y_test, lr_prob)})

# GraphSAGE
_, gs_prob = predict_gnn_probabilities(graphsage_model, graph_data, graph_data.test_mask, config, device)
predictions_dict["GraphSAGE"] = gs_prob
metrics_rows.append({"Model": "GraphSAGE", **compute_classification_metrics(y_test, gs_prob)})

# GAT
_, gat_prob = predict_gnn_probabilities(gat_model, graph_data, graph_data.test_mask, config, device)
predictions_dict["GAT"] = gat_prob
metrics_rows.append({"Model": "GAT", **compute_classification_metrics(y_test, gat_prob)})

# Hybrid Model
hyb_prob = hybrid_model.predict_proba(X_hybrid[test_idx])[:, 1]
predictions_dict["Hybrid"] = hyb_prob
metrics_rows.append({"Model": "Hybrid (SAGE+XGB)", **compute_classification_metrics(y_test, hyb_prob)})

table1_df = pd.DataFrame(metrics_rows)
display(table1_df)

# Export to CSV & LaTeX
export_results_table(table1_df, "table1_classification_metrics", config,
                     caption="Classification Metrics on Held-Out Test Split", label="metrics")

# Plot PR and ROC curves
plot_pr_and_roc_curves(predictions_dict, y_test, config)
plt.show()

## 9. Inference Latency Benchmarking (Table 2 & Figures 4-5)
- Measures Mean $\pm$ Std inference latency (ms) per 1,000 transactions and per single transaction ($N=1$).
- Generates the central Accuracy vs. Latency Trade-Off curve (The Pareto Frontier).
- Decomposes GNN latency into Neighbor Sampling vs. Neural Forward Pass.

In [ ]:
latency_rows = []
gnn_breakdown = {}

sample_1k_tab = X_tab_test[:1000]
sample_1k_neu = X_neu_test[:1000]

# Measure tabular models
xgb_lat_mean, xgb_lat_std = benchmark_tabular_latency(tabular_models["XGBoost"], sample_1k_tab)
rf_lat_mean, rf_lat_std = benchmark_tabular_latency(tabular_models["RandomForest"], sample_1k_tab)
lr_lat_mean, lr_lat_std = benchmark_tabular_latency(tabular_models["LogisticRegression"], sample_1k_neu)
xgb_1_mean, _ = benchmark_tabular_latency(tabular_models["XGBoost"], sample_1k_tab[:1])

# Measure GNN models
gs_lat = benchmark_gnn_latency(graphsage_model, graph_data, test_idx, 1000, config, device)
gnn_breakdown["GraphSAGE"] = gs_lat
gat_lat = benchmark_gnn_latency(gat_model, graph_data, test_idx, 1000, config, device)
gnn_breakdown["GAT"] = gat_lat

latency_rows.append({"Model": "XGBoost", "Latency_per_1k_ms": xgb_lat_mean, "Latency_Std_ms": xgb_lat_std,
                      "Single_tx_ms": xgb_1_mean, "PR-AUC": float(table1_df.loc[table1_df["Model"]=="XGBoost", "PR-AUC"].values[0])})
latency_rows.append({"Model": "RandomForest", "Latency_per_1k_ms": rf_lat_mean, "Latency_Std_ms": rf_lat_std,
                      "Single_tx_ms": float(benchmark_tabular_latency(tabular_models["RandomForest"], sample_1k_tab[:1])[0]),
                      "PR-AUC": float(table1_df.loc[table1_df["Model"]=="RandomForest", "PR-AUC"].values[0])})
latency_rows.append({"Model": "LogisticRegression", "Latency_per_1k_ms": lr_lat_mean, "Latency_Std_ms": lr_lat_std,
                      "Single_tx_ms": float(benchmark_tabular_latency(tabular_models["LogisticRegression"], sample_1k_neu[:1])[0]),
                      "PR-AUC": float(table1_df.loc[table1_df["Model"]=="LogisticRegression", "PR-AUC"].values[0])})
latency_rows.append({"Model": "GraphSAGE", "Latency_per_1k_ms": gs_lat["total_mean"], "Latency_Std_ms": gs_lat["total_std"],
                      "Single_tx_ms": float(benchmark_gnn_latency(graphsage_model, graph_data, test_idx, 1, config, device)["total_mean"]),
                      "PR-AUC": float(table1_df.loc[table1_df["Model"]=="GraphSAGE", "PR-AUC"].values[0])})
latency_rows.append({"Model": "GAT", "Latency_per_1k_ms": gat_lat["total_mean"], "Latency_Std_ms": gat_lat["total_std"],
                      "Single_tx_ms": float(benchmark_gnn_latency(gat_model, graph_data, test_idx, 1, config, device)["total_mean"]),
                      "PR-AUC": float(table1_df.loc[table1_df["Model"]=="GAT", "PR-AUC"].values[0])})

table2_df = pd.DataFrame(latency_rows)
display(table2_df)
export_results_table(table2_df, "table2_latency_benchmark", config,
                     caption="Inference Latency Benchmark across 1,000 Transactions and Single Stream", label="latency")

# Plot Pareto Frontier & Latency Breakdown
plot_accuracy_latency_tradeoff(table2_df, config)
plt.show()

plot_latency_breakdown(gnn_breakdown, config)
plt.show()

## 10. Cold-Start & Isolated Nodes Robustness Analysis (Figure 7)
Evaluates how GraphSAGE degrades on isolated transactions (Degree = 0, new user/card) compared to XGBoost.

In [ ]:
df_robustness = evaluate_cold_start_robustness(tabular_models, graphsage_model, data_dict, graph_data, config, device)
display(df_robustness)
export_results_table(df_robustness, "table3_cold_start_robustness", config,
                     caption="Cold-Start Robustness Comparison: Isolated vs. Connected Nodes", label="cold_start")

plot_cold_start_comparison(df_robustness, config)
plt.show()

## 11. Model Checkpointing & Serialization (Native + ONNX)
Saves all trained weights to disk so they can be reloaded without re-training.

In [ ]:
save_native_models(tabular_models, graphsage_model, gat_model, config, hybrid_model=hybrid_model)
export_tabular_to_onnx(tabular_models, n_features=data_dict["X_tabular"].shape[1], config=config)
print("All models successfully saved!")